# 02: Frozen UNI2 morphology feature grids

Read native histology and supplied transformations, extract registered UNI2-h morphology features, and save calibrated feature grids.

## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import hashlib
import os
import sys
from pathlib import Path

import pandas as pd
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/03_3d").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "00029_native_v2")
OUT = ROOT / "outputs/03_3d" / RUN_ID
DEVICE = os.environ.get(
    "SPAGC_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu"
)
MODE = "path"
FRAME = "00029__g0_registered"
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {".", ".."}:
    raise ValueError("SPAGC_RUN_ID must be a safe directory name")
DATA = ROOT / "data/3d/00029_g0_native"
HE = DATA / "he"
UNI2_CHECKPOINT = ROOT / "models/3d/uni2/pytorch_model.bin"
ARTIFACTS = ROOT / "outputs/03_3d" / RUN_ID
FEATURES = ARTIFACTS / "uni2_features.h5"
if MODE not in ("baseline", "uot", "path"):
    raise ValueError("Unsupported run mode or model mode.")
print({"model_mode": MODE, "output": str(OUT)})
SECTION_RANGE = None
ROI_UM = None
FEATURE_HALO_UM = 112.0
GRID_SPACING_UM = 56.0
PATCH_SIZE_UM = 112.0
BATCH_SIZE = 8
NUCLEUS_SOURCE = DATA / "derived/nucleus_features_registered/nucleus_features.h5"
NUCLEUS_FEATURES = ARTIFACTS / "aligned_nucleus_features.h5"
USE_NUCLEUS_PATH = False
if USE_NUCLEUS_PATH and MODE != "path":
    raise ValueError(
        "The optional supplied-nucleus experiment requires full mode and morphology paths."
    )

In [ ]:
from spagc._io import require_new
from spagc.reconstruction.features import (
    FeatureStore,
    UNI2Encoder,
    align_nucleus_grids,
    precompute_native_section,
)
from spagc.reconstruction.inputs import load_he_manifest

Read the selected native H&E manifest and require the approved local UNI2-h weights. No weight download or substitute encoder is performed.

Read native H&E sections and resolve their supplied calibration, masks and registration metadata.

In [ ]:
if not UNI2_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Supply approved local UNI2-h weights: {UNI2_CHECKPOINT}")
table = load_he_manifest(HE, section_range=SECTION_RANGE, coordinate_frame=FRAME)

## 2. Restore UNI2-h and create the morphology store

Restore the frozen encoder and record its checkpoint identity in the feature provenance. Native patches are read through the supplied registration chain.

Restore the frozen UNI2-h encoder from the supplied local checkpoint and extract morphology vectors.

In [ ]:
store = FeatureStore(require_new(FEATURES), mode="x")
digest = hashlib.sha256()
with UNI2_CHECKPOINT.open("rb") as checkpoint_stream:
    for block in iter(lambda: checkpoint_stream.read(8 * 1024**2), b""):
        digest.update(block)
checkpoint_sha256 = digest.hexdigest()
encoder = UNI2Encoder(str(UNI2_CHECKPOINT), device=DEVICE)

## 3. Extract registered morphology grids

Sample registered grid centres, inverse-map them to native H&E fields of view, and extract pretrained UNI2-h vectors. Store valid locations and coordinate calibration alongside the features.

Read native H&E patches by numerically inverting the supplied registration chain. Apply the crop origin and pixel calibration once.

In [ ]:
feature_qc = []
for row in table.itertuples(index=False):
    record = precompute_native_section(
        row,
        encoder,
        store,
        spacing_um=GRID_SPACING_UM,
        patch_size_um=PATCH_SIZE_UM,
        batch_size=BATCH_SIZE,
        checkpoint_sha256=checkpoint_sha256,
        roi_um=ROI_UM,
        feature_halo_um=FEATURE_HALO_UM,
        coordinate_frame=FRAME,
    )
    feature_qc.append(record)
    print(record, flush=True)

## 4. Check and save feature diagnostics

The extraction step streams features directly to HDF5. Save its coverage table, verify 1,536 feature channels, and align optional nucleus descriptors only when explicitly enabled.

In [ ]:
pd.DataFrame(feature_qc).to_csv(require_new(OUT / "native_feature_qc.csv"), index=False)
assert store.feature_dim == 1536 and len(store.sections) == len(table)
if USE_NUCLEUS_PATH:
    align_nucleus_grids(NUCLEUS_SOURCE, NUCLEUS_FEATURES, store)
print({"feature_dim": store.feature_dim, "sections": len(store.sections)})